## Exporting built-in tools

In [1]:
import sys
print(sys.executable)

c:\Users\hp\OneDrive\Documents\Chatbot_with_LLM\.venv\Scripts\python.exe


In [2]:
import langchain_community
print("Working!")

Working!


C:\Users\hp\AppData\Local\Temp\ipykernel_21316\1489157423.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


### Wikipedia tool

In [48]:
import os

os.environ["USER_AGENT"] = "ChatbotWithLLM/1.0 (dishanichauhan08@gmail.com)"

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=1000
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.


In [11]:
from dotenv import load_dotenv

load_dotenv()

True

### Tavily tool

In [12]:
from langchain_tavily import TavilySearch
tavily_tool=TavilySearch(
    max_results=5,
    topic='general'
)

## Now Custom Tools

In [13]:
from langchain_core.tools import tool

@tool
def add(a: int, b:int)->int:
    """ add a and b"""
    return a+b
@tool
def multiply(a:int,b:int)->int:
    """multiply a and b"""
    return  a*b

In [14]:
tools=[wiki_tool,add,multiply,tavily_tool]

In [33]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "qwen/qwen3.6-27b",
    model_provider="groq",
    max_tokens=500
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001BA3321AD50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BA3321B750>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None, max_tokens=500)

### so we made tools now binding them to the llm is the next step


In [38]:
llm_with_tools=llm.bind_tools(tools)
llm_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001BA3321AD50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BA3321B750>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None, max_tokens=500), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'add', 'description': 'add a and b', 'parameters': {'properties': {'a': {'t

In [28]:
from langchain_core.messages import HumanMessage
query="What is 2*3"
messages = [HumanMessage(query)]
response=llm_with_tools.invoke(query)
print(response)

content='' additional_kwargs={'reasoning_content': "The user is asking for the result of a multiplication operation: 2 * 3.\nI have a `multiply` tool available that takes two integers `a` and `b` and multiplies them.\nI should use this tool to get the answer.\n\nPlan:\n1. Call the `multiply` tool with `a=2` and `b=3`.\n2. Return the result.\n\nLet's call the tool.\n", 'tool_calls': [{'id': '9skt93sm1', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 2058, 'total_tokens': 2189, 'completion_time': 0.250424142, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.162077511, 'prompt_tokens_details': None, 'queue_time': 0.043172345, 'total_time': 0.412501653}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_fff3b79855', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a06bbf-eca

In [18]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': '78sfz35dj',
  'type': 'tool_call'}]

In [19]:
for tool_call in response.tool_calls:
    print(tool_call["name"])
    # name of tools called  yet

multiply


In [20]:
# you can see names of all tools
multiply.name

'multiply'

In [21]:
add.name

'add'

In [22]:
wiki_tool.name

'wikipedia'

In [23]:
tavily_tool.name # these are the names given by us and output is coming as their original name

'tavily_search'

In [29]:

for tool_call in response.tool_calls:
    selected_tool={"add":add, "multiply":multiply,"wikipedia":wiki_tool,"tavily_search": tavily_tool}[tool_call["name"].lower()]
    tool_msg=selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages


[HumanMessage(content='What is 2*3', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='9skt93sm1')]

In [31]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking a simple multiplication question: "What is 2*3".\nI have access to a `multiply` function.\nI should use the `multiply` function to calculate the result.\nThen I will provide the answer.\n\nParameters for `multiply`:\na = 2\nb = 3\n\nI will call `multiply(a=2, b=3)`.\n', 'tool_calls': [{'id': '69f9fnsen', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 2068, 'total_tokens': 2188, 'completion_time': 0.227935236, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.160872766, 'prompt_tokens_details': None, 'queue_time': 0.044362013, 'total_time': 0.388808002}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a06bc3-0630-7b23-8f9d-ff11

In [ ]:
from langchain_core.messages import HumanMessage
query = 'What is langchain and what  is 5*23'
messages=[HumanMessage(query)]
ai_msg = llm_with_tools.invoke(messages)
ai_msg
#llm ne bola ki mjhe ans dene k liye wikipedia tool chaiye

AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking two questions:\n    *   What is "langchain"?\n    *   What is 5 * 23?\n\n2.  **Break Down Tasks:**\n    *   Task 1: Define "langchain". This requires searching or knowing what it is. It\'s likely a software library/framework related to Large Language Models (LLMs). I can use `wikipedia` or `tavily_search` to get a precise definition.\n    *   Task 2: Calculate 5 * 23. I have a `multiply` tool available for this.\n\n3.  **Execute Task 1 (Langchain):**\n    *   Query: "Langchain"\n    *   Tool: `wikipedia` (likely sufficient) or `tavily_search`. Let\'s use `wikipedia` first for a definition.\n\n4.  **Execute Task 2 (Math):**\n    *   Operation: Multiply\n    *   Operand A: 5\n    *   Operand B: 23\n    *   Tool: `multiply`\n\n5.  **Formulate Tool Calls:**\n    *   `wikipedia(query="Langchain")`\n    *   `multiply(a=5, b=23)`\n\n6.  **Synthesize Results:**\n 

In [ ]:
tool_msg = wiki_tool.invoke(tool_call)
tool_msg
#Ye sirf tool ka result hai jo ab hume LLM ko wapas dena hai.

ToolMessage(content="Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.", name='wikipedia', tool_call_id='phnt4k5k7')

In [ ]:
messages.append(tool_msg)
#Ab messages mein Wikipedia ka result bhi save ho gaya.

In [ ]:
ai_msg = llm_with_tools.invoke(messages)
ai_msg
#Ab LLM ko bol rahe hain: "Ye raha Wikipedia ka result. Ab usko multiply tool chiye next query ke liye to vo  bol raha ki multiply tool invoke kro"



AIMessage(content='', additional_kwargs={'reasoning_content': 'I need to answer two parts of the user\'s request:\n1.  Explain what LangChain is.\n2.  Calculate $5 \\times 23$.\n\nFor the first part, I have already retrieved information about LangChain from Wikipedia. The summary says: "LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain\'s use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis."\n\nFor the second part, I need to perform a multiplication. I have a `multiply` tool available.\n$a = 5$\n$b = 23$\n\nI will call the `multiply` tool with these arguments.\n\nFinally, I will combine the information from the Wikipedia summary and the result of the calculation to answer the user\'s question.\n', 'tool_calls': [{'id': 'haqd5vz1z', 'function': {'arguments': '{"a":5,"b":23}

In [ ]:
tool_call = ai_msg.tool_calls[0]

tool_msg = multiply.invoke(tool_call)

print(tool_msg.content)
# invoking multiply tool

115


In [ ]:
messages.append(tool_msg)
# ab humne multiply ka result llm ko dedia ya usme store krdia

In [ ]:
ai_msg = llm_with_tools.invoke(messages)

print(ai_msg.content)
# ab llm ko dubara call kia or ab uske pass dono result the , our final  ans

**LangChain** is a software framework designed to facilitate the integration of large language models (LLMs) into applications. It provides tools and abstractions that make it easier to build applications powered by LLMs, such as chatbots, document analysis tools, summarization engines, and code analysis systems.

Regarding your mathematical question, **5 * 23 is 115**.
